# Milestone 3 - Development Labels and Review Information

Current inputs are `data/labeled/suitability_review_master_v1.csv` for
review information and sampling metadata, and
`data/labeled/suitability_development_labels_v1.csv` for development labels.
Sampling, annotation suggestions and consolidation are already prepared;
older files are retained in `data/labeled/archive/`.
Do not rerun file-generation or annotation-generation code.

Manual verification was not performed for this experiment. The development
dataset combines three recorded first-review decisions with 46 automated
annotations; uncertain pairs are excluded. Evaluation will measure agreement
with these development labels, not independently verified suitability.
Reviewer assignments do not represent completed independent reviews.
This notebook validates the existing inputs and reuses the saved Logistic Regression results. Majority-class, TF-IDF and skill-coverage baselines are reference measurements, not additional trained classifiers. The original split assignments and preprocessing stay fixed. The frozen model is now evaluated on the saved test split in a separate section below. Validation results remain separate.


In [2]:
from pathlib import Path
import pandas as pd

# Find the project from the root or notebooks folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "labeled").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

LABEL_DIR = PROJECT_ROOT / "data" / "labeled"
if not LABEL_DIR.is_dir():
    raise FileNotFoundError(f"Label folder not found: {LABEL_DIR}")


## Historical raw-data inspection — already completed

Optional provenance only; not required for the current workflow.
This saved code is non-executable and was not rerun.

```python
DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Load only the columns needed for Milestone 3.
# Exclude the large precomputed embedding columns.

cv = pd.read_csv(
    DATA_DIR / "cv.csv",
    usecols=[
        "candidate_id",
        "resume_text",
        "clean_skills",
        "inferred_category",
    ],
)

jobs = pd.read_csv(
    DATA_DIR / "job.csv",
    usecols=[
        "job_id",
        "job_title",
        "job_text",
        "clean_skills",
        "category",
    ],
)

matches = pd.read_csv(
    DATA_DIR / "matches.csv",
    usecols=[
        "candidate_id",
        "job_id",
        "label_strength",
    ],
)

print("DATASET SIZES")
print(f"Resumes: {len(cv):,}")
print(f"Jobs: {len(jobs):,}")
print(f"Resume-job pairs: {len(matches):,}")

print("\nMISSING VALUES IN LOADED COLUMNS")
for name, frame in [
    ("Resumes", cv),
    ("Jobs", jobs),
    ("Matches", matches),
]:
    print(f"{name}: {int(frame.isna().sum().sum()):,}")

print("\nDUPLICATE IDENTIFIERS AND PAIRS")
print("Duplicate candidate IDs:", int(cv["candidate_id"].duplicated().sum()))
print("Duplicate job IDs:", int(jobs["job_id"].duplicated().sum()))
print(
    "Duplicate resume-job pairs:",
    int(matches.duplicated(["candidate_id", "job_id"]).sum()),
)

print("\nUNKNOWN REFERENCES")
print(
    "Pairs with unknown candidate IDs:",
    int((~matches["candidate_id"].isin(cv["candidate_id"])).sum()),
)
print(
    "Pairs with unknown job IDs:",
    int((~matches["job_id"].isin(jobs["job_id"])).sum()),
)

print("\nMATCH LABEL DISTRIBUTION")
label_counts = matches["label_strength"].value_counts(dropna=False)

label_summary = pd.DataFrame({
    "Count": label_counts,
    "Percentage": (label_counts / len(matches) * 100).round(2),
})

print(label_summary.to_string())

print("\nCANDIDATE CATEGORY DISTRIBUTION")
print(cv["inferred_category"].value_counts(dropna=False).to_string())

print("\nStep 2 completed. No files were changed.")
```

Historical saved output (not rerun):

```text
DATASET SIZES
Resumes: 9,544
Jobs: 1,167
Resume-job pairs: 122,560

MISSING VALUES IN LOADED COLUMNS
Resumes: 0
Jobs: 0
Matches: 0

DUPLICATE IDENTIFIERS AND PAIRS
Duplicate candidate IDs: 0
Duplicate job IDs: 0
Duplicate resume-job pairs: 0

UNKNOWN REFERENCES
Pairs with unknown candidate IDs: 0
Pairs with unknown job IDs: 0

MATCH LABEL DISTRIBUTION
                 Count  Percentage
label_strength                    
medium_positive  70636       57.63
weak_positive    51150       41.73
strong_positive    774        0.63

CANDIDATE CATEGORY DISTRIBUTION
inferred_category
INFORMATION-TECHNOLOGY    8140
FINANCE                   1010
SALES                      215
BUSINESS-DEVELOPMENT        83
HR                          80
UNKNOWN                     16

Step 2 completed. No files were changed.
```


In [3]:
# Load prepared inputs; preserve blanks and identifier strings.
review_path = LABEL_DIR / "suitability_review_master_v1.csv"
development_path = LABEL_DIR / "suitability_development_labels_v1.csv"

for file_path in [review_path, development_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Required input not found: {file_path}")

review = pd.read_csv(review_path, keep_default_na=False, dtype=str)
training_data = pd.read_csv(
    development_path, keep_default_na=False, dtype=str
)

for name, frame in [("Master review", review), ("Development", training_data)]:
    if frame["pair_id"].duplicated().any():
        raise ValueError(f"Duplicate pair IDs in {name}.")
    expected_ids = frame["candidate_id"] + ":" + frame["job_id"]
    if not frame["pair_id"].eq(expected_ids).all():
        raise ValueError(f"Conflicting identifiers in {name}.")

if not training_data["pair_id"].isin(review["pair_id"]).all():
    raise ValueError("Development pairs are missing from the master.")
if not training_data["training_label"].isin(["0", "1"]).all():
    raise ValueError("Development labels must be binary; U is not 0.")
if not training_data["label_source"].isin([
    "existing_final_label", "recorded_first_review", "automated_annotation"
]).all():
    raise ValueError("Unknown development label source.")
for column in ["resume_text", "job_text"]:
    if training_data[column].str.strip().eq("").any():
        raise ValueError(f"Empty development document: {column}")

# Sampling metadata comes directly from the master.
sampling_columns = [
    "pair_id", "candidate_id", "job_id", "sampling_group",
    "sampling_domain", "sampling_reason", "candidate_inferred_domain",
    "source_match_recorded", "source_label_strength", "selection_version",
]
sampling = review[sampling_columns].copy()
if sampling["sampling_domain"].str.strip().eq("").any():
    raise ValueError("Missing sampling domain in the master.")

shared_columns = [
    "candidate_id", "job_id", "resume_text", "job_title",
    "job_text", "sampling_domain",
]
expected = review.set_index("pair_id").loc[
    training_data["pair_id"], shared_columns
]
actual = training_data.set_index("pair_id")[shared_columns]
if not actual.equals(expected):
    raise ValueError("Development documents or metadata differ from the master.")

reviewer_present = review["reviewer_1"].str.strip().ne("")
label_present = review["label_1"].str.strip().ne("")
untouched = ~reviewer_present & ~label_present
uncertain = reviewer_present & review["label_1"].str.strip().isin(["", "U"])

print("Master review pairs:", len(review))
print("Development pairs:", len(training_data))
print("Excluded from development:", len(review) - len(training_data))
print("Recorded first reviewers:", int(reviewer_present.sum()))
print("Uncertain first reviews:", int(uncertain.sum()))
print("Pairs without a first-review entry:", int(untouched.sum()))
print("Development label sources:")
print(training_data["label_source"].value_counts().to_string())
print("Automated annotations are not completed human reviews.")

# Summarize the prepared development labels.
label_counts = training_data["training_label"].value_counts()
print("\nDevelopment label counts:")
print("Suitable (1):", int(label_counts.get("1", 0)))
print("Unsuitable (0):", int(label_counts.get("0", 0)))

domain_counts = pd.crosstab(
    training_data["sampling_domain"], training_data["training_label"]
).reindex(columns=["1", "0"], fill_value=0)
domain_counts = domain_counts.rename(columns={"1": "Suitable", "0": "Unsuitable"})
domain_counts["Total"] = domain_counts.sum(axis=1)
print("\nDevelopment label counts by domain:")
print(domain_counts.to_string())


Master review pairs: 150
Development pairs: 49
Excluded from development: 101
Recorded first reviewers: 4
Uncertain first reviews: 1
Pairs without a first-review entry: 146
Development label sources:
label_source
automated_annotation     46
recorded_first_review     3
Automated annotations are not completed human reviews.

Development label counts:
Suitable (1): 12
Unsuitable (0): 37

Development label counts by domain:
training_label          Suitable  Unsuitable  Total
sampling_domain                                    
BUSINESS-DEVELOPMENT           1          13     14
FINANCE                        2           5      7
HR                             4           3      7
INFORMATION-TECHNOLOGY         5           9     14
SALES                          0           7      7


## Suitability Annotation Rubric

### Purpose

Assign a suitability label to each resume–job pair using evidence from
the resume and job description.

### Label Definitions

**1 — Suitable Match**

The resume demonstrates the core skills and relevant experience needed
for the role, with no clearly unmet essential requirement. Relevant
transferable skills may count when the reviewer explains the equivalence.
Missing optional or preferred skills alone do not make a pair unsuitable.

**0 — Unsuitable Match**

The resume shows a substantial mismatch with the role's core duties or
clearly does not meet an essential requirement. The reviewer must identify
the specific requirement and supporting evidence.

**Uncertain**

Leave the label blank when information is insufficient, contradictory,
or ambiguous. Record the reason and keep the pair pending. An omitted
skill phrase alone does not prove that the candidate lacks the skill.

### Review Rules

- Read the full resume and job description.
- Assess job-relevant skills, responsibilities, and experience.
- Distinguish essential requirements from optional preferences.
- Do not use personal or protected characteristics.
- Do not use source match strengths, similarity scores, or skill-coverage
  percentages as ground-truth suitability labels.
- Do not assume that an unrecorded pair is unsuitable.
- Record a short evidence-based rationale for each decision.

### Annotation method used in this experiment

The development dataset combines three recorded first-review
decisions with 46 automated annotations. Independent human
verification was not performed. Uncertain pairs are excluded.
Evaluation will measure agreement with these development labels;
it will not establish independently verified suitability.

### Interpretation

Labels describe suitability under this rubric. They do not represent
actual hiring decisions or probabilities of being hired.

## Existing splits and features

We retain the reviewed v2 train, validation and test assignments. Candidate IDs, normalized resumes, job IDs and normalized job texts have no overlap across splits. This workflow does not regenerate splits or fit preprocessing.

The four features remain TF-IDF cosine similarity, mentioned-job-skill coverage, missing-job-skill count and an unavailable-skill-evidence indicator. We retain the saved training-fitted TF-IDF, dictionary and feature order.

## One classifier and reference baselines

Logistic Regression is the only active trained classifier. Its saved pipeline uses StandardScaler, balanced class weights and C=10. A score greater than or equal to 0.5 predicts suitable. We do not retrain or tune settings here.

The majority-class, TF-IDF threshold and skill-coverage threshold results remain reference measurements from the earlier validation run. They are not additional trained classifiers. The active table contains these three baselines and Logistic Regression only.

## Reusing reviewed results

The module checks original dataset hashes, package versions and all recorded v2 artifact hashes before loading results. It reports the current consumer-code fingerprint separately and checks inputs and sources again before returning. Historical reports and artifacts, including the earlier Random Forest comparison, remain unchanged.

Future exports and website inference must use `load_logistic_regression()` and the fixed threshold 0.5. That loader verifies the saved classifier, scaling and feature order. Browser inference is not implemented here.

In [4]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.milestone3 import run
comparison, split_audit, selected_model = run(PROJECT_ROOT)
display(comparison)
print("Selected trained model:", selected_model)

Reused v2: existing reviewed v2 results; original training provenance preserved. Current consumer-code SHA256: 25dcf0af320d4b734631d0bce994a233365adceb4d0a30d64858118435ccb6ca


,name,settings,threshold,macro_f1,accuracy,confusion_matrix,unsuitable_precision,unsuitable_recall,unsuitable_f1,unsuitable_support,suitable_precision,suitable_recall,suitable_f1,suitable_support
0,logistic_regression,{'C': 10.0},0.5,1.000000,1.0,"[[7, 0], [0, 3]]",1.000000,1.000000,1.000000,7,1.0,1.000000,1.000000,3
1,majority,{'training_majority': 0},0.5,0.411765,0.7,"[[7, 0], [3, 0]]",0.700000,1.000000,0.823529,7,0.0,0.000000,0.000000,3
2,tfidf_threshold,{},0.1,0.696970,0.7,"[[4, 3], [0, 3]]",1.000000,0.571429,0.727273,7,0.5,1.000000,0.666667,3
3,skill_threshold,{},0.3,0.687500,0.8,"[[7, 0], [2, 1]]",0.777778,1.000000,0.875000,7,1.0,0.333333,0.500000,3


Selected trained model: {'model': 'logistic_regression', 'settings': {'C': 10.0}, 'threshold': 0.5, 'rule': 'score >= threshold predicts suitable', 'refit_on_train_plus_validation': False}


## Held-out test evaluation

We load the reviewed v2 manifest and saved Logistic Regression, TF-IDF, dictionary, aliases and feature order. The threshold stays 0.5. Test features use transform only; the model and baseline settings were fixed before inspecting test results.

The evaluation saves versioned test features, predictions with label sources, per-class metrics, baseline measurements, domain coverage, error analysis and a labeled confusion matrix. Identical completed outputs are verified and reused without recomputing predictions. Current evaluation-code fingerprints remain separate from historical training fingerprints.

In [5]:
from src.milestone3_evaluation import evaluate_test, load_assets, analyze_skill_gap
evaluation_folder, test_comparison = evaluate_test(PROJECT_ROOT)
print("Held-out test outputs:", evaluation_folder)
display(test_comparison)

Reused completed test evaluation v2; no predictions recomputed.
Held-out test outputs: D:\Downloads\depi_gapAnalyzer\reports\milestone3\test_evaluation\v2


,name,macro_f1,accuracy,confusion_matrix,unsuitable_precision,unsuitable_recall,unsuitable_f1,unsuitable_support,suitable_precision,suitable_recall,suitable_f1,suitable_support
0,logistic_regression,0.761905,0.8,"[[6, 2], [0, 2]]",1.00,0.750,0.857143,8,0.500000,1.0,0.666667,2
1,majority,0.444444,0.8,"[[8, 0], [2, 0]]",0.80,1.000,0.888889,8,0.000000,0.0,0.000000,2
2,tfidf_threshold,0.866667,0.9,"[[7, 1], [0, 2]]",1.00,0.875,0.933333,8,0.666667,1.0,0.800000,2
3,skill_threshold,0.375000,0.6,"[[6, 2], [2, 0]]",0.75,0.750,0.750000,8,0.000000,0.0,0.000000,2


## Errors and domain coverage

The report records false positives and false negatives with pair IDs, labels, titles, domains and feature values. Document observations suggest possible causes rather than proving them. We keep all labels, settings and model artifacts unchanged after seeing test results.

Finance has no test examples. Unsupported domains and absent classes are shown explicitly; their performance is not treated as zero.

In [6]:
import json
print((evaluation_folder / "test_evaluation_report.md").read_text(encoding="utf8"))
display(pd.read_csv(evaluation_folder / "test_errors.csv"))
print(json.dumps(json.loads((evaluation_folder / "domain_results.json").read_text()), indent=2))

# Held-out test evaluation v2

The frozen Logistic Regression correctly classified 8 of ten test pairs. Test macro F1 is 0.7619, compared with 1.0000 on validation. We retained C=10, StandardScaler, balanced class weights and threshold 0.5. No training, refitting, tuning or split regeneration occurred. Scores at least 0.5 predict Suitable Match; the positive class index was read from classes_.

## Results and reference baselines

               name  macro_f1  accuracy confusion_matrix  unsuitable_precision  unsuitable_recall  unsuitable_f1  unsuitable_support  suitable_precision  suitable_recall  suitable_f1  suitable_support
logistic_regression  0.761905       0.8 [[6, 2], [0, 2]]                  1.00              0.750       0.857143                   8            0.500000              1.0     0.666667                 2
           majority  0.444444       0.8 [[8, 0], [2, 0]]                  0.80              1.000       0.888889                   8            0.000000            

,pair_id,job_title,sampling_domain,training_label,label_source,logistic_regression_score,logistic_regression_prediction,majority_prediction,tfidf_threshold_prediction,skill_threshold_prediction,tfidf_cosine,job_skill_coverage,missing_job_skill_count,no_job_skills
0,cand_01689:3899539354,Senior Vice President Human Resources,HR,0,automated_annotation,0.657754,1,0,0,0,0.058692,0.0,3,0
1,cand_02542:3901981623,Information Technology Administrator,INFORMATION-TECHNOLOGY,0,automated_annotation,0.673582,1,0,0,0,0.086435,0.0,5,0


[
  {
    "domain": "BUSINESS-DEVELOPMENT",
    "pairs": 3,
    "suitable": 1,
    "unsuitable": 2,
    "metrics": {
      "macro_f1": 1.0,
      "accuracy": 1.0,
      "confusion_matrix": [
        [
          2,
          0
        ],
        [
          0,
          1
        ]
      ],
      "unsuitable_precision": 1.0,
      "unsuitable_recall": 1.0,
      "unsuitable_f1": 1.0,
      "unsuitable_support": 2,
      "suitable_precision": 1.0,
      "suitable_recall": 1.0,
      "suitable_f1": 1.0,
      "suitable_support": 1
    },
    "note": "Descriptive only; absent classes: []. Class metrics with zero support are undefined, represented by zero under zero_division=0."
  },
  {
    "domain": "FINANCE",
    "pairs": 0,
    "suitable": 0,
    "unsuitable": 0,
    "metrics": null,
    "note": "Unsupported: no test examples"
  },
  {
    "domain": "HR",
    "pairs": 2,
    "suitable": 1,
    "unsuitable": 1,
    "metrics": {
      "macro_f1": 0.3333333333333333,
      "accuracy": 0.5,

## Synthetic skill-gap feedback example

This example is invented for demonstration and is outside the evaluation dataset. Feedback uses the same saved preprocessing and classifier. It lists mentioned job skills not found in the resume and gives conditional suggestions. Coverage is on a 0?1 scale, or null when no job skills are recognized. A classifier score is not a hiring probability.

In [7]:
assets = load_assets(PROJECT_ROOT)
synthetic_feedback = analyze_skill_gap(
    "I use Python and JS to build small applications.",
    "The role mentions Python, JavaScript, SQL and Node.js.",
    assets,
)
print(json.dumps(synthetic_feedback, indent=2, allow_nan=False))

Reused v2: existing reviewed v2 results; original training provenance preserved. Current consumer-code SHA256: 25dcf0af320d4b734631d0bce994a233365adceb4d0a30d64858118435ccb6ca


Reused v2: existing reviewed v2 results; original training provenance preserved. Current consumer-code SHA256: 25dcf0af320d4b734631d0bce994a233365adceb4d0a30d64858118435ccb6ca


{
  "resume_skills": [
    "javascript",
    "python"
  ],
  "job_skills": [
    "javascript",
    "node.js",
    "python",
    "sql"
  ],
  "matched_skills": [
    "javascript",
    "python"
  ],
  "job_skills_not_found": [
    "node.js",
    "sql"
  ],
  "job_skill_coverage": 0.5,
  "coverage_scale": "0\u20131; null means insufficient extracted job-skill evidence",
  "tfidf_text_similarity": 0.13493731621900448,
  "model_classification": "Suitable Match",
  "classifier_score": 0.9676818549122053,
  "classification_threshold": 0.5,
  "score_note": "A classifier score is not a hiring probability. Classification, skill coverage and text similarity are separate outputs.",
  "skill_evidence_note": "These are mentioned skills, not necessarily essential requirements. A skill not found in your resume may still be one you have.",
  "suggestions": [
    {
      "skill": "node.js",
      "guidance": "node.js appears in the job description but was not found in your resume. If you have used node.

## Limitations and remaining work

The test set contains only ten pairs and two suitable examples, with no Finance examples. Most development labels are automated and have not been independently verified, so metrics measure agreement with annotations. Sales has no suitable development examples, and all five suitable IT examples share one help-desk job. Domain-level results are descriptive and uncertain.

Logistic Regression remains the only trained classifier even though the fixed TF-IDF baseline scores higher on test. Browser export and React integration remain future work. Milestone 3 is still in progress.